# Data preprocessing
The collected building data consists of either 15-minute interval time-series data or event-based COV (Change of Value) data.
In addtion, all measurements are stored from the most recent record backward up to 1 year. Each data point is saved as an individual file.


However, experts and simulators in the building energy domain typically work with data arranged chronologically from past to present.
Therfore, to support this workflow, the dataset is reoranized so that the originally reverse-ordered records are converted into past to present time series.
Both 15-munite interval data and the COV dtat are aligned into a unified 15-minute interval time-series format, after which all data points are consolicated into a single file.

# 0. Setting

In [7]:
from pathlib import Path
import re
import pandas as pd

In [8]:
DATA_ROOT = Path("/Users/kim-yujin/Desktop/CSC_Mapping/Dataset")

TIME_COL = "Timestamp"
VALUE_COL = "Value"

GLOBAL_START = pd.Timestamp("2025-01-01 00:00:00")
GLOBAL_END   = pd.Timestamp("2025-10-01 00:00:00")
TIME_FREQ    = "15T"   # 15 minute interval

# 1. Utility functions

In [9]:
def load_excel_safe(path: Path) -> pd.DataFrame:
    """
    read excel file:
      - first column: Timestamp
      - second column: Value
    (current file format: 'Time stamp', 'AHU-2 SaFanASts')
    """
    df = pd.read_excel(path)

    cols = list(df.columns)
    if len(cols) < 2:
        raise ValueError(f"{path.name}: columns less than 2.")

    if TIME_COL not in df.columns:
        df.rename(columns={cols[0]: TIME_COL}, inplace=True)
    if VALUE_COL not in df.columns:
        df.rename(columns={cols[1]: VALUE_COL}, inplace=True)

    return df[[TIME_COL, VALUE_COL]]


def to_ts_index(df: pd.DataFrame) -> pd.DataFrame:
    """
    transformate Timestamp to datetime,
    arrange past to current, set index.
    """
    df = df.copy()
    df[TIME_COL] = pd.to_datetime(df[TIME_COL])
    df = df.sort_values(TIME_COL, ascending=True)
    df.set_index(TIME_COL, inplace=True)
    df[VALUE_COL] = pd.to_numeric(df[VALUE_COL], errors="coerce")
    return df


def detect_series_type(df: pd.DataFrame) -> str:
    """
    detect interval based type:
      - if median interval is 15 minute and std is low -> 15min
      - else -> COV
    (arrange)
    """
    ts = pd.to_datetime(df[TIME_COL]).sort_values(ascending=True)
    deltas = ts.diff().dropna().dt.total_seconds()

    if deltas.empty:
        return "unknown"

    median = deltas.median()
    std = deltas.std()

    # 15 min (900sec) ± 1min, low interval std -> 15 min time-series
    if abs(median - 900) < 60 and (pd.isna(std) or std < 300):
        return "15min"
    else:
        return "cov"


def get_base_point_name(stem: str) -> str:
    """
    file name modify: 'data point name'.

    example:
      'AHU-2 SaTmp_15min_Jan' -> 'AHU-2 SaTmp'
      'AHU-2 SaTmp_COV_Feb'   -> 'AHU-2 SaTmp'
      'AHU-2 SaFanASts'       -> 'AHU-2 SaFanASts'
    """
    pattern = r"(_?15min|_?15m|_?15|_?cov|_?COV)"
    cleaned = re.sub(pattern, "", stem)
    return cleaned.strip()

# 2. Process point level

In [10]:
def process_point_group(point_name: str, file_paths: list[Path]) -> pd.Series | None:
    """
    process same point(several files shared base name):
      - classify files as 15min or COV
      - merge 15min files → 15 min resample(mean)
      - merge COV files → 15 min resample(ffill)
      - merge both 15min and COV files as combine_first -> final 15 min series
    """
    print(f"\n Process point: {point_name}")
    fifteen_dfs = []
    cov_dfs = []

    # 1) file type
    for f in file_paths:
        print(f"  file: {f.name}")
        try:
            df_raw = load_excel_safe(f)
        except Exception as e:
            print(f"    ! failed load: {e}")
            continue

        name_lower = f.name.lower()

        # if filen ame contains 'cov' -> COV
        if "cov" in name_lower:
            stype = "cov"
        # if file name contains '15' and not cov -> assume 15 min series
        elif ("15" in name_lower) and ("cov" not in name_lower):
            stype = "15min"
        else:
            # else assume interval base
            stype = detect_series_type(df_raw)

        print(f"    > data type: {stype}")

        if stype == "15min":
            fifteen_dfs.append(df_raw)
        elif stype == "cov":
            cov_dfs.append(df_raw)
        else:
            print("    ! type unknown, ignore this file")

    series_15 = None
    series_cov = None

    # 2) merge 15 min time series
    if fifteen_dfs:
        df15 = pd.concat([to_ts_index(df) for df in fifteen_dfs])
        df15 = df15.sort_index()
        series_15 = df15[VALUE_COL].resample(TIME_FREQ).mean()
        print(f"    v number of merged 15min point: {len(series_15)}")

    # 3) merge COV (what ever current to pass, arrange at to_ts_index)
    if cov_dfs:
        df_cov = pd.concat([to_ts_index(df) for df in cov_dfs])
        df_cov = df_cov.sort_index()
        series_cov = df_cov[VALUE_COL].resample(TIME_FREQ).ffill()
        print(f"    ✓ number of merged COV -> 15min point: {len(series_cov)}")

    # 4) merge two type series
    if series_15 is not None and series_cov is not None:
        idx = series_15.index.union(series_cov.index)
        s15 = series_15.reindex(idx)
        scov = series_cov.reindex(idx)
        combined = s15.combine_first(scov)
    elif series_15 is not None:
        combined = series_15
    elif series_cov is not None:
        combined = series_cov
    else:
        print("    ! No time series, return None")
        return None

    combined.name = point_name
    return combined

# 3. Process entire AHU-2 folder

In [11]:
def build_building_timeseries(data_root: Path) -> pd.DataFrame:
    # 1) grouping files as base point name
    groups: dict[str, list[Path]] = {}

    for f in sorted(data_root.glob("*.xlsx")):
        # ignore last runed result CSV
        if f.name.startswith("Building_AllPoints_"):
            continue

        stem = f.stem
        base_name = get_base_point_name(stem)
        groups.setdefault(base_name, []).append(f)

    print(f"all point number (base name): {len(groups)}")

    # 2) process every point group
    point_series: dict[str, pd.Series] = {}
    for base_name, files in groups.items():
        s = process_point_group(base_name, files)
        if s is not None:
            point_series[base_name] = s

    if not point_series:
        raise RuntimeError("X all points not processed. Check file structure.")

    # 3) apply same time line (15 min)
    time_index = pd.date_range(GLOBAL_START, GLOBAL_END, freq=TIME_FREQ)
    print("\nsame time line")
    print(f"  start: {time_index[0]}")
    print(f"  end  : {time_index[-1]}")
    print(f"  number: {len(time_index)}")

    aligned = {}
    for name, s in point_series.items():
        s_aligned = s.reindex(time_index)
        s_aligned = s_aligned.ffill()
        aligned[name] = s_aligned

    df = pd.DataFrame(aligned, index=time_index)
    df.index.name = TIME_COL

    print("\n complete integration")
    print(f"  shape = {df.shape} (row: timestep, column: point)")
    return df

# 4. Run main

In [12]:
if __name__ == "__main__":
    df = build_building_timeseries(DATA_ROOT)

    output_path = DATA_ROOT / "Building_AllPoints_15m_2025Jan01_2025Oct01_COV_and_15min.csv"
    df.to_csv(output_path)
    print(f"\nComplete saving: {output_path}")


all point number (base name): 15

 Process point: AHU-2 AvgCcoilTmp
  file: AHU-2 AvgCcoilTmp.xlsx


/var/folders/35/_0fhx4x939d01fyjcy5t9jtw0000gn/T/ipykernel_78248/873191391.py:42: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ts = pd.to_datetime(df[TIME_COL]).sort_values(ascending=True)


    > data type: 15min


/var/folders/35/_0fhx4x939d01fyjcy5t9jtw0000gn/T/ipykernel_78248/873191391.py:28: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[TIME_COL] = pd.to_datetime(df[TIME_COL])
/var/folders/35/_0fhx4x939d01fyjcy5t9jtw0000gn/T/ipykernel_78248/2043986127.py:50: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  series_15 = df15[VALUE_COL].resample(TIME_FREQ).mean()


    v number of merged 15min point: 34391

 Process point: AHU-2 AvgMaTmp
  file: AHU-2 AvgMaTmp.xlsx


...

/var/folders/35/_0fhx4x939d01fyjcy5t9jtw0000gn/T/ipykernel_78248/873191391.py:42: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ts = pd.to_datetime(df[TIME_COL]).sort_values(ascending=True)


    > data type: 15min


/var/folders/35/_0fhx4x939d01fyjcy5t9jtw0000gn/T/ipykernel_78248/873191391.py:28: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[TIME_COL] = pd.to_datetime(df[TIME_COL])


    v number of merged 15min point: 34390

same time line
  start: 2025-01-01 00:00:00
  end  : 2025-10-01 00:00:00
  number: 26209

 complete integration
  shape = (26209, 15) (row: timestep, column: point)

Complete saving: /Users/kim-yujin/Desktop/CSC_Mapping/Dataset/Building_AllPoints_15m_2025Jan01_2025Oct01_COV_and_15min.csv


/var/folders/35/_0fhx4x939d01fyjcy5t9jtw0000gn/T/ipykernel_78248/2043986127.py:50: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  series_15 = df15[VALUE_COL].resample(TIME_FREQ).mean()
/var/folders/35/_0fhx4x939d01fyjcy5t9jtw0000gn/T/ipykernel_78248/2782542463.py:27: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  time_index = pd.date_range(GLOBAL_START, GLOBAL_END, freq=TIME_FREQ)
